# 第 4 周 Day 6：控制频率、推理延迟与重规划 — Notebook 作业

[← Week 04 / Day 05](day-05.ipynb) · [本周 Notebook](README.md) · [课程正文](../../week-04/day-06.md) · [Goal 进度](../../PROGRESS.md) · [Week 04 / Day 07 →](day-07.ipynb)

> 状态：**未提交**。直接编辑各个 Markdown/Code 单元格；“教练验收区”不要预填。


## Goal

预计 100–130 分钟。把模型推理频率、环境控制频率、chunk horizon、重规划间隔和延迟统一到时间轴，完成至少三种调度配置的公平闭环实验。

### 我的目标复述

【双击此 Markdown 单元格，用自己的话填写今日目标及其在 VLA 中的作用。】


## Setup

| 字段 | 我的记录 |
|---|---|
| 实际投入时间 | 【填写】 |
| 完成日期 | 【填写】 |
| Python / PyTorch | 【填写；纯理论日写“不适用”】 |
| CPU / GPU / 仿真器 | 【填写】 |
| 资源等级 | 【L0 / L1 / L2】 |
| 产物路径 | 【填写】 |

生成时环境检查（2026-09-01）：当前可见 Python 未检测到 Jupyter、ipykernel、nbformat、PyTorch 或 NumPy。本 Notebook 已做结构验证，但在该环境中尚未执行。


In [ ]:
# 可选：Notebook 环境可用后运行此单元，记录基础环境。
import platform
import sys

print("python:", sys.version)
print("platform:", platform.platform())


## Context：知识点及其在 VLA 中的作用

Action chunk 既是学习目标也是系统调度选择。若控制器 20 Hz、chunk 8 步，则覆盖 0.4 s；模型推理 150 ms 时，每步重规划可能跟不上。系统可执行旧 chunk、保持上个动作或降低规划频率，每种选择影响陈旧性、平滑度和恢复能力。


## Concepts：概念、公式、形状与数据流

定义：控制周期 `dt_c=1/f_c`；规划周期 `dt_p=k/f_c`（每 k 个控制步规划）；chunk 覆盖 `T_chunk=H_a/f_c`；观测到动作生效总延迟 `tau`。

```text
control ticks: 0 1 2 3 4 5 ...
plan id A:     predict | execute A0 A1 A2 ...
plan id B:                 predict | execute B0 ...
```

需要记录每个执行动作的 `(control_t, source_observation_t, prediction_id, chunk_offset, latency)`。平滑度可用 `mean ||a_t-a_{t-1}||`，但不能替代成功率与安全指标。


## Learning Steps

1. 对 `f_c=20 Hz,H_a=8,k=1/4/8` 计算规划率与覆盖时间。
2. 设计推理耗时小于/大于 `dt_c` 时的调度策略。
3. 实现 scheduler 日志，不需要真实异步线程。
4. 在同一 seeds 上比较三组 k 或延迟。
5. 汇总成功率、最终误差、动作平滑度、陈旧观测 age。


## Steps：必做作业

### 课程题目

基于 Day 4/5 的规则或学习策略，固定控制 20 Hz、`H_a=4`，比较 `k=1,2,4`；模拟推理延迟 0 和 2 control steps，并在固定时刻注入扰动。至少 10 seeds（周项目扩为 20）。提交时间轴、每配置指标和一条逐步调度日志，解释哪种配置更适合当前结果，不预设赢家。

下面每道题都有独立作答单元。文字、表格、公式或 Mermaid 写在 Markdown 单元；可运行代码写在后面的 Code 单元。


### 第 1 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 第 2 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 第 3 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 可运行代码 / 实验区

纯理论日可以保留为空；代码日请将实现拆成短小单元，并保留关键输出。


In [ ]:
# 在此编写或运行当天代码。
# 建议先写清输入 shape、dtype、设备和随机种子。


## Checks：输入、预期输出与验证

- 输入：同一初态/扰动 seeds、`f_c=20,H_a=4,k∈{1,2,4},delay∈{0,2}`。
- 预期：每配置有 rollout 表；每执行动作可追溯 prediction id/offset；时间单位一致。
- 验证：`T_chunk=0.2s`；k=4 的规划率为 5 Hz；延迟索引不读取未来预测；比较使用相同 horizon/成功阈值。

低资源方案：只用解析/noisy chunk policy，避免训练和真实 sleep；离散步模拟延迟。

### 我的验证记录

| 检查项 | 实际结果 | 是否符合 | 证据 |
|---|---|---|---|
| 输入 shape / schema | 【填写】 | 【填写】 | 【填写】 |
| 输出 shape / schema | 【填写】 | 【填写】 | 【填写】 |
| dtype、范围和单位 | 【填写】 | 【填写】 | 【填写】 |
| 正向测试 | 【填写】 | 【填写】 | 【填写】 |
| 负向测试 / 错误注入 | 【填写】 | 【填写】 | 【填写】 |
| 指标分子 / 分母 / seed | 【填写】 | 【填写】 | 【填写】 |

> 尚未运行的内容必须标为“预期结果”，不能作为实际证据。


In [ ]:
# 在此编写 shape、dtype、数值范围、断言或负向测试。


## Evidence：提交与复现证据

固定包含：`时序参数表`、`调度策略`、`一条逐步日志`、`10 seeds 汇总`、`扰动/延迟结果`、`成功与平滑度取舍`、`投入分钟数`。

### 我的证据

- 代码路径：【填写】
- 配置路径：【填写】
- 数据 / checkpoint / commit 或哈希：【填写】
- 实际命令：【填写】
- 退出码：【填写】
- 关键输出：【填写】
- 结果说明了什么：【填写】
- 结果没有说明什么：【填写】
- 失败现象与定位证据：【填写】

### VLA 约束

| 约束 | 我的定义 |
|---|---|
| 图像布局、颜色顺序和范围 | 【填写】 |
| 文本 token、padding 与 mask | 【填写】 |
| 机器人状态各维含义 | 【填写】 |
| 坐标系、长度和角度单位 | 【填写】 |
| 动作空间及逐维定义 | 【填写】 |
| observation/action 时间对齐 | 【填写】 |
| 控制频率 / action chunk | 【填写】 |
| 归一化及统计量来源 | 【填写】 |
| 随机种子与数据划分 | 【填写】 |


## Self-check：课程自测

1. 20 Hz 下 8 步 chunk 覆盖多久？
2. 重规划间隔 k 与 chunk 长度为何可不同？
3. 观测 age 与推理 latency 有何区别？
4. 更平滑为何不一定更成功？


### 自测第 1 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 自测第 2 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 自测第 3 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 自测第 4 题作答

【双击此 Markdown 单元格，在这里填写答案。】


## Help：排查、最低完成线与提高

### 常见错误

- 把 H_a 当秒数：除以控制频率才是覆盖秒数。
- 模拟延迟用真实 sleep 导致不可复现：教学实验优先离散队列。
- k=1 时预测覆盖旧计划但执行索引仍加 1：新 plan offset 应从 0。
- 平滑动作掩盖不达目标：同时报告成功和最终距离。

### 最低完成线

完成所有时间换算，一条 k=1 与一条 k=4 调度日志，并比较无延迟/2步延迟各一次。

### 可选提高

模拟异步 inference queue：新预测晚到时丢弃过时 plan 或从对应 offset 接入，讨论两种策略的安全边界。


## Rubric

100 分，80 分通过：时间计算 20；scheduler 25；日志可追溯 15；公平实验 20；指标/解释 20。规划率计算错、延迟实现偷看未来、不同配置换 seed/成功标准均为关键失败。

### 提交前检查

- [ ] 已逐项完成必做作业；
- [ ] 已区分实际结果与预期结果；
- [ ] 已保留代码输出、日志、表格或推理证据；
- [ ] 已记录适用的 shape、坐标系、单位、动作和时间约定；
- [ ] 已完成验证或明确写出无法执行的原因；
- [ ] 已回答全部自测题；
- [ ] 已记录仍不确定的点或失败案例。


## Coach Review（学习者请勿填写）

| 字段 | 验收结果 |
|---|---|
| 证据完整性 | 待验收 |
| Rubric 得分 | /100 |
| 门槛项 | 待验收 |
| 当天状态 | 未提交 |
| 具体缺口 |  |
| 最小补救任务 |  |
| 复验结果 |  |
| 下一课程 |  |


## Next Steps

完成后保存 Notebook，并把路径发到学习对话：

`docs/vla-learning/notebooks/week-04/day-06.ipynb`

教练验收通过后才会更新 `PROGRESS.md`。

[← Week 04 / Day 05](day-05.ipynb) · [本周 Notebook](README.md) · [课程正文](../../week-04/day-06.md) · [Week 04 / Day 07 →](day-07.ipynb)
